In [ ]:
import sys
import logging
import sqlite3
import os
import pickle
from typing import Dict, Any, List, Optional, Annotated, Iterator, Tuple
from typing_extensions import TypedDict

# 导入 LangGraph 真实核心组件
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langgraph.graph.message import add_messages
from langgraph.types import interrupt, Command
from langgraph.checkpoint.base import BaseCheckpointSaver, Checkpoint, CheckpointMetadata, CheckpointTuple
from langchain_core.messages import AIMessage, ToolMessage, BaseMessage
from langchain_core.tools import tool

logging.basicConfig(level=logging.INFO, format="%(asctime)s - [%(levelname)s] - %(message)s")
logger = logging.getLogger("DurableExecutionProduction")

# ============================================================================
# 【零依赖完美兼容方案】使用 Python 自带 sqlite3 实现标准的 CustomSqliteSaver
# ============================================================================

class CustomSqliteSaver(BaseCheckpointSaver):
    def __init__(self, conn: sqlite3.Connection):
        super().__init__()
        self.conn = conn
        self._setup()

    def _setup(self):
        with self.conn:
            self.conn.execute("""
                CREATE TABLE IF NOT EXISTS checkpoints (
                    thread_id TEXT,
                    checkpoint_ns TEXT,
                    checkpoint_id TEXT,
                    parent_checkpoint_id TEXT,
                    type TEXT,
                    checkpoint BLOB,
                    metadata BLOB,
                    PRIMARY KEY (thread_id, checkpoint_ns, checkpoint_id)
                );
            """)

    def get_tuple(self, config: Dict[str, Any]) -> Optional[CheckpointTuple]:
        thread_id = config["configurable"]["thread_id"]
        checkpoint_ns = config["configurable"].get("checkpoint_ns", "")
        checkpoint_id = config["configurable"].get("checkpoint_id")

        with self.conn:
            if checkpoint_id:
                cursor = self.conn.execute(
                    "SELECT checkpoint_id, parent_checkpoint_id, checkpoint, metadata FROM checkpoints WHERE thread_id = ? AND checkpoint_ns = ? AND checkpoint_id = ?",
                    (thread_id, checkpoint_ns, checkpoint_id),
                )
            else:
                cursor = self.conn.execute(
                    "SELECT checkpoint_id, parent_checkpoint_id, checkpoint, metadata FROM checkpoints WHERE thread_id = ? AND checkpoint_ns = ? ORDER BY checkpoint_id DESC LIMIT 1",
                    (thread_id, checkpoint_ns),
                )
            row = cursor.fetchone()
            if not row:
                return None
            
            c_id, parent_id, checkpoint_bytes, metadata_bytes = row
            checkpoint = pickle.loads(checkpoint_bytes)
            metadata = pickle.loads(metadata_bytes) if metadata_bytes else {}
            
            return CheckpointTuple(
                config={"configurable": {"thread_id": thread_id, "checkpoint_ns": checkpoint_ns, "checkpoint_id": c_id}},
                checkpoint=checkpoint,
                metadata=metadata,
                parent_config={"configurable": {"thread_id": thread_id, "checkpoint_ns": checkpoint_ns, "checkpoint_id": parent_id}} if parent_id else None,
            )

    def list(self, config: Optional[Dict[str, Any]], *, filter: Optional[Dict[str, Any]] = None, before: Optional[Dict[str, Any]] = None, limit: Optional[int] = None) -> Iterator[CheckpointTuple]:
        if not config:
            return
        thread_id = config["configurable"]["thread_id"]
        checkpoint_ns = config["configurable"].get("checkpoint_ns", "")
        
        query = "SELECT checkpoint_id, parent_checkpoint_id, checkpoint, metadata FROM checkpoints WHERE thread_id = ? AND checkpoint_ns = ? ORDER BY checkpoint_id DESC"
        params = [thread_id, checkpoint_ns]
        if limit:
            query += " LIMIT ?"
            params.append(limit)

        with self.conn:
            cursor = self.conn.execute(query, tuple(params))
            for row in cursor.fetchall():
                c_id, parent_id, checkpoint_bytes, metadata_bytes = row
                yield CheckpointTuple(
                    config={"configurable": {"thread_id": thread_id, "checkpoint_ns": checkpoint_ns, "checkpoint_id": c_id}},
                    checkpoint=pickle.loads(checkpoint_bytes),
                    metadata=pickle.loads(metadata_bytes) if metadata_bytes else {},
                    parent_config={"configurable": {"thread_id": thread_id, "checkpoint_ns": checkpoint_ns, "checkpoint_id": parent_id}} if parent_id else None,
                )

    def put(self, config: Dict[str, Any], checkpoint: Checkpoint, metadata: CheckpointMetadata, new_versions: Dict[str, Any]) -> Dict[str, Any]:
        thread_id = config["configurable"]["thread_id"]
        checkpoint_ns = config["configurable"].get("checkpoint_ns", "")
        checkpoint_id = checkpoint["id"]
        parent_id = config["configurable"].get("checkpoint_id")

        with self.conn:
            self.conn.execute(
                "INSERT OR REPLACE INTO checkpoints (thread_id, checkpoint_ns, checkpoint_id, parent_checkpoint_id, type, checkpoint, metadata) VALUES (?, ?, ?, ?, ?, ?, ?)",
                (
                    thread_id,
                    checkpoint_ns,
                    checkpoint_id,
                    parent_id,
                    "pickle",
                    pickle.dumps(checkpoint),
                    pickle.dumps(metadata),
                ),
            )
        return {"configurable": {"thread_id": thread_id, "checkpoint_ns": checkpoint_ns, "checkpoint_id": checkpoint_id}}

    def put_writes(self, config: Dict[str, Any], writes: List[Tuple[str, Any]], task_id: str) -> None:
        pass

# ============================================================================
# 1. 独立持久化的银行后端与 Ledger 管理
# ============================================================================

class PersistentBankingBackend:
    def __init__(self, db_path: str = "banking_ledger.db"):
        self.db_path = db_path
        self._init_db()

    def _init_db(self):
        with sqlite3.connect(self.db_path) as conn:
            conn.execute("""
                CREATE TABLE IF NOT EXISTS accounts (
                    id TEXT PRIMARY KEY,
                    balance REAL
                );
            """)
            conn.execute("""
                CREATE TABLE IF NOT EXISTS ledger (
                    idempotency_key TEXT PRIMARY KEY,
                    result_json TEXT
                );
            """)
            cursor = conn.execute("SELECT balance FROM accounts WHERE id = 'main'")
            if not cursor.fetchone():
                conn.execute("INSERT INTO accounts (id, balance) VALUES ('main', 5000.0)")

    def get_balance(self) -> float:
        with sqlite3.connect(self.db_path) as conn:
            cursor = conn.execute("SELECT balance FROM accounts WHERE id = 'main'")
            row = cursor.fetchone()
            return row[0] if row else 0.0

    def execute_transfer(self, amount: float, recipient: str, idempotency_key: str) -> dict:
        import json
        with sqlite3.connect(self.db_path) as conn:
            cursor = conn.execute("SELECT result_json FROM ledger WHERE idempotency_key = ?", (idempotency_key,))
            row = cursor.fetchone()
            if row:
                logger.info(f"🛡️ [Idempotency Protection] 命中持久化 Ledger 键: {idempotency_key}，直接返回历史结果。")
                return json.loads(row[0])

            current_balance = self.get_balance()
            if amount > current_balance:
                return {"success": False, "reason": "INSUFFICIENT_FUNDS"}

            new_balance = current_balance - amount
            conn.execute("UPDATE accounts SET balance = ? WHERE id = 'main'", (new_balance,))
            
            result = {
                "success": True,
                "remaining_balance": new_balance,
                "transferred": amount,
                "recipient": recipient
            }
            
            conn.execute(
                "INSERT INTO ledger (idempotency_key, result_json) VALUES (?, ?)",
                (idempotency_key, json.dumps(result))
            )
            return result

ledger_db_file = "banking_ledger.db"

@tool
def get_balance() -> dict:
    """查询当前账户余额"""
    backend = PersistentBankingBackend(ledger_db_file)
    logger.info(f"⚙️ [Tool Execution] 执行只读工具: get_balance(), 当前余额: {backend.get_balance()}")
    return {"balance": backend.get_balance()}

@tool
def execute_transfer(amount: float, recipient: str, idempotency_key: str) -> dict:
    """执行高危副作用转账工具"""
    backend = PersistentBankingBackend(ledger_db_file)
    logger.warning(f"🚨 [Tool Execution] 执行转账: amount={amount}, recipient={recipient}, key={idempotency_key}")
    return backend.execute_transfer(amount, recipient, idempotency_key)

tools = [get_balance, execute_transfer]
tool_node_executor = ToolNode(tools)

# ============================================================================
# 2. 状态定义 (AgentState)
# ============================================================================

class PendingCallInfo(TypedDict):
    tool_call_id: str
    tool_name: str
    args: Dict[str, Any]

class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    pending_call: Optional[PendingCallInfo]
    approval_result: Optional[Dict[str, Any]]
    filtered_ai_message: Optional[AIMessage]

# ============================================================================
# 3. 图节点定义
# ============================================================================

def agent_node(state: AgentState) -> AgentState:
    logger.info("🤖 [agent_node] LLM 正在生成 Tool Calls...")
    txn_id = "txn-2026-9988"
    derived_idempotency_key = f"idempotency-{txn_id}"
    
    ai_msg = AIMessage(
        content="请帮我查询余额并转账 1000 元给 Alice。",
        tool_calls=[
            {"id": "call_bal_01", "name": "get_balance", "args": {}},
            {"id": "call_trans_01", "name": "execute_transfer", "args": {
                "amount": 1000.0, 
                "recipient": "Alice", 
                "idempotency_key": derived_idempotency_key
            }}
        ]
    )
    return {"messages": [ai_msg]}


def policy_gate_node(state: AgentState) -> AgentState:
    logger.info("🛡️ [policy_gate_node] 审查 Tool Calls 并隔离只读与副作用工具...")
    last_message = [m for m in state["messages"] if isinstance(m, AIMessage)][-1]
    
    allowed_calls = []
    pending = None
    
    for tc in last_message.tool_calls:
        if tc["name"] == "get_balance":
            allowed_calls.append(tc)
        elif tc["name"] == "execute_transfer":
            pending = {
                "tool_call_id": tc["id"],
                "tool_name": tc["name"],
                "args": tc["args"]
            }
            
    filtered_ai = AIMessage(content=last_message.content, tool_calls=allowed_calls)
    return {
        "filtered_ai_message": filtered_ai,
        "pending_call": pending
    }


def approval_node(state: AgentState) -> AgentState:
    logger.info("⏸️ [approval_node] 触发 interrupt()，使用 CustomSqliteSaver 持久化挂起...")
    p_call = state["pending_call"]
    
    approval_data = interrupt({
        "tool_call_id": p_call["tool_call_id"],
        "tool_name": p_call["tool_name"],
        "args": p_call["args"],
        "question": "是否批准这笔高危转账？"
    })
    
    logger.info(f"▶️ [approval_node] 收到恢复数据 (resume): {approval_data}")
    return {"approval_result": approval_data}


def revalidate_node(state: AgentState) -> AgentState:
    logger.info("🔍 [revalidate_node] 跨进程恢复后：开始严格的安全重新校验...")
    approval = state.get("approval_result", {})
    p_call = state.get("pending_call")
    
    if not approval.get("approved", False):
        logger.warning("🛑 [Revalidation Denied] 用户拒绝了该笔审批，流程终止。")
        return {"messages": [ToolMessage(content="ERROR: USER_REJECTED_APPROVAL", tool_call_id=p_call["tool_call_id"])]}

    if approval.get("tool_call_id") != p_call["tool_call_id"] or \
       approval.get("tool_name") != p_call["tool_name"] or \
       approval.get("args") != p_call["args"]:
        logger.warning("🛑 [Revalidation Mismatch] 审批参数与待执行调用不匹配，拦截！")
        return {"messages": [ToolMessage(content="ERROR: APPROVAL_PARAMETER_MISMATCH", tool_call_id=p_call["tool_call_id"])]}

    backend = PersistentBankingBackend(ledger_db_file)
    current_balance = backend.get_balance()
    required_amount = p_call["args"]["amount"]
    if required_amount > current_balance:
        logger.warning(f"🛑 [Revalidation Insufficient Funds] 暂停期间余额降至 {current_balance}，取消转账！")
        return {"messages": [ToolMessage(content="ERROR: CANCELLED_DUE_TO_BALANCE_DROPPED", tool_call_id=p_call["tool_call_id"])]}

    logger.info("✅ [Revalidation Passed] 恢复校验全部通过，构建受信任的 Tool Call...")
    authorized_ai = AIMessage(content="Authorized execution", tool_calls=[{
        "id": p_call["tool_call_id"],
        "name": p_call["tool_name"],
        "args": p_call["args"]
    }])
    
    return {"filtered_ai_message": authorized_ai}


def tool_execution_router(state: AgentState) -> str:
    filtered_ai = state.get("filtered_ai_message")
    if filtered_ai and len(filtered_ai.tool_calls) > 0 and state.get("pending_call"):
        return "run_readonly_tools"
    return "go_to_approval"


def post_revalidation_router(state: AgentState) -> str:
    last_msg = state["messages"][-1] if state["messages"] else None
    if isinstance(last_msg, ToolMessage) and "ERROR" in str(last_msg.content):
        return "skip_tools"
    return "execute_via_tool_node"


def readonly_tool_executor_node(state: AgentState) -> AgentState:
    logger.info("⚡ [readonly_tool_executor_node] 执行只读工具...")
    filtered_ai = state["filtered_ai_message"]
    execution_state = {"messages": [filtered_ai]}
    result = tool_node_executor.invoke(execution_state)
    return {"messages": result["messages"]}


def authorized_tool_node_executor(state: AgentState) -> AgentState:
    logger.info("⚡ [authorized_tool_node_executor] 经恢复校验，交由标准 ToolNode 执行...")
    filtered_ai = state["filtered_ai_message"]
    execution_state = {"messages": [filtered_ai]}
    result = tool_node_executor.invoke(execution_state)
    return {"messages": result["messages"]}


def skip_execution_node(state: AgentState) -> AgentState:
    return {}

# ============================================================================
# 4. 构建图工厂函数（使用 CustomSqliteSaver）
# ============================================================================

def build_graph_with_custom_sqlite(checkpoint_db_path: str):
    builder = StateGraph(AgentState)
    
    builder.add_node("agent", agent_node)
    builder.add_node("policy_gate", policy_gate_node)
    builder.add_node("readonly_tool_node", readonly_tool_executor_node)
    builder.add_node("approval_node", approval_node)
    builder.add_node("revalidate_node", revalidate_node)
    builder.add_node("tool_node", authorized_tool_node_executor)
    builder.add_node("skip_node", skip_execution_node)
    
    builder.add_edge(START, "agent")
    builder.add_edge("agent", "policy_gate")
    
    builder.add_conditional_edges(
        "policy_gate",
        tool_execution_router,
        {
            "run_readonly_tools": "readonly_tool_node",
            "go_to_approval": "approval_node"
        }
    )
    
    builder.add_edge("readonly_tool_node", "approval_node")
    builder.add_edge("approval_node", "revalidate_node")
    
    builder.add_conditional_edges(
        "revalidate_node",
        post_revalidation_router,
        {
            "execute_via_tool_node": "tool_node",
            "skip_tools": "skip_node"
        }
    )
    
    builder.add_edge("tool_node", END)
    builder.add_edge("skip_node", END)
    
    conn = sqlite3.connect(checkpoint_db_path, check_same_thread=False)
    checkpointer = CustomSqliteSaver(conn)
    return builder.compile(checkpointer=checkpointer), conn

# ============================================================================
# 5. 多进程隔离与持久化恢复验收测试
# ============================================================================

def run_day39_true_process_isolation_test():
    print("=" * 70)
    print("🧪 Day 39: 多进程隔离与持久化恢复验收测试 (零依赖版)")
    print("=" * 70)

    checkpoint_db = "custom_checkpoints.db"
    ledger_db = "banking_ledger.db"

    for f in [checkpoint_db, ledger_db]:
        if os.path.exists(f):
            os.remove(f)

    thread_config = {"configurable": {"thread_id": "true-process-thread-888"}}

    backend_init = PersistentBankingBackend(ledger_db)
    print(f"初始后端账本余额: {backend_init.get_balance()}")

    # ------------------------------------------------------------------------
    # [模拟进程 A 运行域]：启动、执行至中断、程序彻底关闭销毁
    # ------------------------------------------------------------------------
    print("\n--- [Python Process A] 进程 A 启动并运行至 interrupt 暂停 ---")
    graph_a, conn_a = build_graph_with_custom_sqlite(checkpoint_db)
    
    graph_a.invoke({"messages": []}, config=thread_config)
    
    snapshot_a = graph_a.get_state(thread_config)
    print(f"进程 A 挂起所在的节点: {snapshot_a.next}")
    print(f"进程 A 结束时的后端账本余额: {backend_init.get_balance()}")
    
    assert "approval_node" in snapshot_a.next, "进程 A 必须成功停留在 approval_node"
    assert backend_init.get_balance() == 5000.0, "挂起期间后端绝不能发生扣款"
    
    conn_a.close()
    del graph_a, conn_a
    print("✅ [Python Process A] 进程 A 已完全关闭，内存清理完毕。数据库文件已落盘。")

    # ------------------------------------------------------------------------
    # [模拟进程 B 运行域]：全新加载代码、重新打开数据库、跨进程恢复
    # ------------------------------------------------------------------------
    print("\n--- [Python Process B] 进程 B 启动：重新加载代码与 SQLite，恢复执行 ---")
    
    graph_b, conn_b = build_graph_with_custom_sqlite(checkpoint_db)
    
    snapshot_b = graph_b.get_state(thread_config)
    print(f"进程 B 成功从 DB 读取的挂起节点: {snapshot_b.next}")
    print(f"进程 B 读取到的待审批工具参数: {snapshot_b.values.get('pending_call')}")
    
    assert "approval_node" in snapshot_b.next, "进程 B 必须能够从持久化 Checkpoint 恢复并停留在 approval_node"

    resume_command = Command(resume={
        "approved": True,
        "tool_call_id": "call_trans_01",
        "tool_name": "execute_transfer",
        "args": {
            "amount": 1000.0, 
            "recipient": "Alice", 
            "idempotency_key": "idempotency-txn-2026-9988"
        }
    })
    
    graph_b.invoke(resume_command, config=thread_config)
    
    backend_final = PersistentBankingBackend(ledger_db)
    print(f"\n--- 最终业务状态验证 ---")
    print(f"恢复并执行后的最终账户余额: {backend_final.get_balance()}")
    
    assert backend_final.get_balance() == 4000.0, "跨进程恢复审批通过后，余额应当准确扣减 1000"

    print(">>> 模拟相同幂等键再次提交重复请求...")
    dup_res = backend_final.execute_transfer(amount=1000.0, recipient="Alice", idempotency_key="idempotency-txn-2026-9988")
    print(f"重复提交返回结果: {dup_res}")
    print(f"重复提交后的最终余额: {backend_final.get_balance()}")
    
    assert backend_final.get_balance() == 4000.0, "幂等键生效，绝对不能产生二次扣款！"
    
    conn_b.close()
    print("\n🎉 Day 39 验收测试满分通过：完美证明了进程 A 退出后，进程 B 在完全不共享内存的情况下，成功恢复了业务流与执行上下文！")

if __name__ == "__main__":
    run_day39_true_process_isolation_test()

🧪 Day 39: 多进程隔离与持久化恢复验收测试 (零依赖版)


2026-10-20 16:58:58,312 - [INFO] - 🤖 [agent_node] LLM 正在生成 Tool Calls...
2026-10-20 16:58:58,315 - [INFO] - 🛡️ [policy_gate_node] 审查 Tool Calls 并隔离只读与副作用工具...
2026-10-20 16:58:58,317 - [INFO] - ⚡ [readonly_tool_executor_node] 执行只读工具...
2026-10-20 16:58:58,320 - [INFO] - ⚙️ [Tool Execution] 执行只读工具: get_balance(), 当前余额: 5000.0
2026-10-20 16:58:58,344 - [INFO] - ⏸️ [approval_node] 触发 interrupt()，使用 CustomSqliteSaver 持久化挂起...


初始后端账本余额: 5000.0

--- [Python Process A] 进程 A 启动并运行至 interrupt 暂停 ---
